In [1]:
!pip install langchain langchain-community faiss-cpu sentence-transformers

In [2]:
# ================================
# 1. Load Text File
# ================================
with open('meditations copy.text', "r", encoding="utf-8") as f:
    data = f.read()

print("Total text length:", len(data))

Total text length: 343721


In [3]:



# ================================
# 2. Chunking (Sentence-based)
# ================================
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize

def split_into_sentence_chunks(text, max_chunk_length=500):
    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > max_chunk_length:
            chunks.append(current_chunk)
            current_chunk = sentence
        else:
            current_chunk += " " + sentence

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

sentence_chunks = split_into_sentence_chunks(data, 500)

# Clean chunks (remove noise + very small chunks)
sentence_chunks = [chunk.strip() for chunk in sentence_chunks if len(chunk.strip()) > 100]

# Remove headings like "MEDITATIONS"
sentence_chunks = [chunk for chunk in sentence_chunks if "MEDITATIONS" not in chunk[:50]]

print("Total cleaned chunks:", len(sentence_chunks))







[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total cleaned chunks: 821


In [4]:
# ================================
# 3. Embeddings (Offline)
# ================================
from sentence_transformers import SentenceTransformer



In [5]:


from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use same embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create vector store directly from text
vectorstore = FAISS.from_texts(sentence_chunks, embedding_model)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

/tmp/ipykernel_8304/2747921121.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
import re

def clean_text(text):
    # remove headings
    text = re.sub(r"MEDITATIONS OF MARCUS AURELIUS.*", "", text)
    text = re.sub(r"Marcus Aurelius.*Casaubon.*", "", text)
    text = re.sub(r"uploaded to .*", "", text)
    text = re.sub(r"Page\s*\d+.*", "", text)

    # remove book titles like "THIRD BOOK", "NINTH BOOK"
    text = re.sub(r"\b[A-Z]+\s+BOOK\b", "", text)

    # clean extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [22]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline


from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=120,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=pipe)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [23]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
Use the following context to answer the question clearly.

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [8]:
!pip install -U langchain

In [26]:
query = "What does Marcus Aurelius say about controlling the mind?"

docs = retriever.invoke(query)

# Take best chunk only
best_chunk = docs[0].page_content
cleaned = clean_text(best_chunk)

print("\nAnswer:\n")
print(cleaned[:400])


Answer:

He remembers besides that whatsoever partakes of reason, is akin unto him, and that to care for all men generally, is agreeing to the nature of a man: but as for honour and praise, that they ought not generally to be admitted and accepted of from all, but from such only, who live according to nature.
